## WE CAN ALSO SKIP THIS NB AND USE THE DATAFRAME IDS THAT WE HAVE ALREADY HAVE

In [ ]:
import requests
import json
import base64

from vantage6.client import UserClient

In [ ]:
# This is standard authentication Keycloak flow. @Itziar; we need to discuss on how
# to deal with this for the demo. We could create a token that is valid for 10 years
# and use that token to authenticate?
client = UserClient(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es:443/server",
    auth_url="https://vantage6-auth.orchestrator.idea.lst.tfo.upm.es:443",
    auth_client="public_client",
    auth_realm="vantage6",
    log_level="INFO"
)
# You can authenticate using the user `itziar` and the password that I've send to you.
client.authenticate()

# Set the headers for the other requests
headers = {
    "Authorization": f"Bearer {client._access_token}"
}

# Print the server version
print("Server version: ", client.util.get_server_version())

In [ ]:
#
# Static content
#
image = "harbor2.vantage6.ai/idea4rc/sessions:latest"
method = "create_cohort"

# Organization IDs for the test collaboration with FAKE OMOP data
UPM_ORG_ID = 3
IKNL_ORG_ID = 1
ORG_IDS = [UPM_ORG_ID, IKNL_ORG_ID]

# Collaboration ID for the test collaboration with FAKE OMOP data
COLLABORATION_ID = 2

# Demo session ID
SESSION_ID = 2

#
# Dynamic content
#
# NOTE:
#   - `omop` is the FAKE OMOP database when using the test collaboration
#   - `int` is the BlueBerry data
label = "omop"

# NOTE:
#   The features indicate which SQL query to use to extract the patient features. The
#   options are:
#   - `sarcoma` - for the IDEA4RC Sarcoma data
#   - `int` - for the BlueBerry data
#   - `head_neck` - for the IDEA4RC Head and Neck data NOT IMPLEMENTED YET
features = "sarcoma"

# NOTE:
#   The patient ids are depending on which data source. In the case of the FAKE OMOP
#   data we have 3 cohorts; RPS, Pelvis and RPS+Pelvis. Each organization has its own
#   set of patient ids, so we need to make sure to send the right ids to the right
#   organization. So this is specific for the FAKE OMOP data.
with open("./pts_lists.json", "r") as f:
    pts_list = json.load(f)

# In the demo the IDs of all patients in each center are the same, so we do not need
# to send different IDs to different organizations. In the real scenario we need to
# send the right IDs to the right organization.
patient_ids_rps = pts_list["pts_RPS"]
# NOTE:
# It is also possible to create the other cohorts:
# - patient_ids_pelvis = pts_list["pts_Pelvis"]
# - patient_ids_pelvis_rps = pts_list["pts_RPS_Pelvis"]

# Each organization (can) receive a different input
orgs_input = [
    {
        "id": id_,
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "patient_ids": patient_ids,
                    "features": features
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
    for id_, patient_ids in zip(ORG_IDS, [patient_ids_rps]*len(ORG_IDS))
]

In [ ]:
payload = {
    "label": label,
    # "name": name, # optional, v6 will generate a name if not provided
    "task": {
        "method": method,
        "image": image,
        # In vantage6 we can (but we dont in IDEA4RC) use end-to-end encryption,
        # therefore we need to store the input for each organization individually.
        "organizations": orgs_input
    }
}

In [ ]:
# Create a vantage6 task to extract the data from the OMOP data source and store it
# into a dataframe.
response = requests.post(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/{SESSION_ID}/dataframe",
    headers=headers,
    json=payload
)
TASK_ID = response.json()["last_session_task"]["id"]
DATAFRAME_ID = response.json()["id"]
response.json()

In [ ]:
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
response.json()